In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.formula.api import logit
from scipy.stats import chi2_contingency

from pathlib import Path
import pickle
import os

#### Project Question
* Does region, payment method, age, or gender also predict customer churn?

In [2]:
# Import dataset
root = Path.cwd().parent
dir_path = Path.cwd()
dataset = root/'data'/'customer_churn_clean.csv'
results_dir = f'{dir_path}/results/'

df = pd.read_csv(dataset, index_col=0)
df.head()

,customer_id,gender,region,subscription_plan,payment_method,age,days_since_last_login,customer_service_calls,monthly_spend,churn
0,10001,Male,East,Basic,PayPal,27.0,27,1.0,61.08,0
1,10002,NaN,West,Pro,PayPal,55.0,146,2.0,73.09,0
2,10003,Female,West,Pro,E-Wallet,NaN,147,2.0,116.30,1
3,10004,Female,South,Basic,Credit Card,NaN,34,1.0,0.58,1
4,10005,Male,North,Basic,Crypto,30.0,146,3.0,46.35,1


In [3]:
# Understanding data columns, dtypes, and observation count 
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             10000 non-null  int64  
 1   gender                  9039 non-null   str    
 2   region                  10000 non-null  str    
 3   subscription_plan       10000 non-null  str    
 4   payment_method          9300 non-null   str    
 5   age                     8491 non-null   float64
 6   days_since_last_login   10000 non-null  int64  
 7   customer_service_calls  9950 non-null   float64
 8   monthly_spend           9441 non-null   float64
 9   churn                   10000 non-null  int64  
dtypes: float64(3), int64(3), str(4)
memory usage: 781.4 KB


#### Analysis Variables
* Region
    - Geographic territory
* Gender
    - Reported customer gender
* Payment Method
    - Most frequent payment type
* Age
    - Customer's age in years

**_Objectives_**
* Identify the relationship between analysis variables and churn
* Perform chi-squared test of independece to all variables to determine if any have a statistically significant relationship with churn
* Conduct a one-tail ANOVA test on significant variables in relation to churn
* Perform Post-Hoc Validation of ANOVA test using Tukey's HSD (Honesty Significant Difference)

In [4]:
# Subsetting the dataset
subset = df[['region','gender','payment_method','age','churn']].dropna().copy()
subset.age = subset.age.astype(int)
subset.head()

,region,gender,payment_method,age,churn
0,East,Male,PayPal,27,0
4,North,Male,Crypto,30,1
5,West,Male,Crypto,46,1
6,South,Male,E-Wallet,22,1
7,East,Non-Binary,E-Wallet,44,1


In [5]:
def save_result(path:str,filename:str,data):
    with open(path+filename+'.pkl', 'wb') as to_write:
        pickle.dump(data,to_write)

def read_result(path:str,filename:str):
    with open(path+filename+'.pkl', 'rb') as to_read:
        results_data = pickle.load(to_read)
    return results_data

def multi_chi_test(columns,data,target,alpha=0.05,results_dir=None,test_name=None):
    rows = []
    for col in columns:
        observed = pd.crosstab(data[target],data[col])
        res = chi2_contingency(observed)

        rows.append({
            'Model':f'{target} ~ {col}',
            'Chi2 Value':(res.statistic).round(2),
            'Significance Level':alpha,
            'P-value':res.pvalue,
            'Significant':res.pvalue<alpha,
            'Degrees of Freedom':res.dof
        })
    results = pd.DataFrame(rows)

    if results_dir and test_name:
        save_result(results_dir,test_name,results)
results = read_result(results_dir,'chi_square_test_results')
results

,Model,Chi2 Value,Significance Level,P-value,Significant,Degrees of Freedom
0,Churn ~ region,31.65,0.05,0.00000062,True,3
0,Churn ~ gender,0.07,0.05,0.96620874,False,2
0,Churn ~ payment_method,4.65,0.05,0.19904529,False,3
0,Churn ~ age,57.34,0.05,0.22158986,False,50
